In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
SEED = 42
torch.set_num_threads(4)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

In [ ]:
def find_project_root():

    current = Path.cwd().resolve()

    while current != current.parent:

        if (current / ".git").exists():
            return current

        current = current.parent

    raise FileNotFoundError(
        "Could not find project root."
    )


PROJECT_ROOT = find_project_root()

DATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "gtsrb"
)

SPLIT_DIR = (
    PROJECT_ROOT
    / "splits"
)

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

In [ ]:
from Common.dataset import create_dataloaders

from Common.models import (
    ModelB,
    count_parameters
)

from Common.training import (
    train_model
)

In [ ]:
train_df = pd.read_csv(
    SPLIT_DIR / "train_split.csv"
)

val_df = pd.read_csv(
    SPLIT_DIR / "val_split.csv"
)

test_df = pd.read_csv(
    SPLIT_DIR / "test_split.csv"
)

In [ ]:
train_loader, val_loader, test_loader = (
    create_dataloaders(
        train_df,
        val_df,
        test_df,
        DATA_ROOT,
        batch_size=128,
        num_workers=0
    )
)

In [ ]:
model = ModelB().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer_configs = {

    "Adam": lambda parameters:
        optim.Adam(
            parameters,
            lr=1e-3
        ),

    "SGD": lambda parameters:
        optim.SGD(
            parameters,
            lr=1e-2
        ),

    "SGD_Momentum": lambda parameters:
        optim.SGD(
            parameters,
            lr=1e-2,
            momentum=0.9
        )
}

In [ ]:
optimizer_results = {}

for name, optimizer_function in optimizer_configs.items():

    print("\n" + "=" * 50)
    print(f"Training with {name}")
    print("=" * 50)

    # Reset random seeds for a fair comparison
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    # Reset the shared loader shuffle so each optimizer sees the same sample order.
    train_loader.generator.manual_seed(SEED)

    # Every optimizer gets a completely fresh Model B
    model = ModelB().to(device)

    optimizer = optimizer_function(
        model.parameters()
    )

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        epochs=20,
        device=device
    )

    optimizer_results[name] = history

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for name, history in optimizer_results.items():

    plt.plot(
        range(1, len(history["val_accuracy"]) + 1),
        history["val_accuracy"],
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Optimizer Comparison - Validation Accuracy")

plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for name, history in optimizer_results.items():

    plt.plot(
        range(1, len(history["val_loss"]) + 1),
        history["val_loss"],
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Optimizer Comparison - Validation Loss")

plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
optimizer_summary = []

for name, history in optimizer_results.items():

    best_val_acc = max(
        history["val_accuracy"]
    )

    final_val_acc = history[
        "val_accuracy"
    ][-1]

    final_val_loss = history[
        "val_loss"
    ][-1]

    avg_epoch_time = np.mean(
        history["epoch_time"]
    )

    optimizer_summary.append({
        "Optimizer": name,
        "Best Validation Accuracy": best_val_acc,
        "Final Validation Accuracy": final_val_acc,
        "Final Validation Loss": final_val_loss,
        "Average Epoch Time (s)": avg_epoch_time
    })


optimizer_df = pd.DataFrame(
    optimizer_summary
)

optimizer_df

In [ ]:
# Display optimizer comparison
print(optimizer_df)

# Create result folders if they do not already exist
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
METRICS_DIR = RESULTS_DIR / "metrics"

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save optimizer comparison table
optimizer_df.to_csv(
    TABLES_DIR / "optimizer_comparison.csv",
    index=False
)

print(
    "Optimizer comparison saved to:",
    TABLES_DIR / "optimizer_comparison.csv"
)

### Momentum interpretation

The SGD with Momentum run uses a momentum coefficient of 0.9. Momentum accumulates a decaying history of gradients, which can speed updates along directions that consistently reduce the loss and damp oscillation across steep directions. It can also overshoot minima when the learning rate and momentum are too large. Compare the measured validation curves, accuracy, and loss for SGD and SGD with Momentum after the three runs finish; this notebook does not yet contain completed measurements.


Selecting Final Optimizer

In [ ]:
ranked_optimizers = optimizer_df.sort_values(
    by=[
        "Best Validation Accuracy",
        "Final Validation Loss"
    ],
    ascending=[
        False,
        True
    ]
)

print("Optimizer ranking:")
display(ranked_optimizers)

In [ ]:
selected_optimizer = ranked_optimizers.iloc[0][
    "Optimizer"
]

print(
    "Selected optimizer:",
    selected_optimizer
)

In [ ]:
selected_optimizer_path = (
    METRICS_DIR
    / "selected_optimizer.txt"
)

selected_optimizer_path.write_text(
    str(selected_optimizer)
)

print(
    "Selected optimizer saved as:",
    selected_optimizer
)

print(
    "File:",
    selected_optimizer_path
)